<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/Feature%20Extraction%20El%C5%91tan%C3%ADtott%20Modellekkel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Extraction Előtanított Modellekkel

Az **előtanított konvolúciós neurális hálózatok** (pretrained CNNs) az egyik legfontosabb eszközök a modern computer vision-ben. Ezek a modellek milliónyi képen tanultak meg vizuális jellemzőket felismerni, és ezt a tudást **transzferálhatjuk** saját feladatainkra.

A **feature extraction** (jellemző-kinyerés) azt jelenti, hogy egy előtanított hálózatot "jellemző-generátorként" használunk: a képet beadjuk a hálózatnak, és valamelyik belső réteg kimenetét vesszük ki mint a kép **numerikus reprezentációját**.

## Miért működik ez?

A CNN-ek hierarchikus jellemzőket tanulnak:
- **Korai rétegek**: Élek, sarkok, egyszerű textúrák
- **Középső rétegek**: Összetettebb minták, textúra kombinációk
- **Késői rétegek**: Objektum-részek, szemantikus jellemzők

Ezek a jellemzők **általánosan hasznosak** - nem csak az eredeti feladatra (pl. ImageNet osztályozás), hanem más vision feladatokra is!

## Tartalomjegyzék

1. Feature Extraction alapjai és motiváció
2. Hook-ok használata PyTorch-ban
3. Különböző rétegek jellemzőinek vizsgálata
4. Jellemző-vektorok összehasonlítása
5. Gyakorlati alkalmazások
6. Többrétegű jellemző-kinyerés

# Feature Extraction előtanított modellekkel

Az **előtanított CNN-ek** (ResNet, VGG, EfficientNet) nem csak osztályozásra használhatók - a belső rétegeik **általános vizuális jellemzőket** tanultak meg, amelyeket más feladatokra is felhasználhatunk.

## Miért működik?

A CNN rétegek hierarchikusan tanulnak:
- **Korai rétegek**: Élek, textúrák, egyszerű minták
- **Késői rétegek**: Objektumrészek, szemantikus jellemzők

Ezek a jellemzők **transzferálhatók** más feladatokra!

## Tartalomjegyzék

1. Feature extraction alapok
2. Hook-ok használata PyTorch-ban
3. Jellemző-vektorok összehasonlítása
4. Különböző rétegek vizsgálata
5. Gyakorlati alkalmazások

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
np.random.seed(42)
torch.manual_seed(42)

## 1. Feature extraction alapok

### A CNN mint jellemző-hierarchia

```
Kép (224×224×3)
    ↓ [Conv rétegek]
Feature Maps (csökkenő méret, növekvő csatornaszám)
    ↓ [Global Average Pool]
Feature Vector (512 vagy 2048 dim)
    ↓ [Fully Connected]
Osztály predikció (1000 ImageNet osztály)
```

**Feature extraction**: A feature vector-t használjuk, nem a végső predikciót!

In [ ]:
# ResNet18 struktúra
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("ResNet18 rétegei:")
print("=" * 40)
for name, module in resnet18.named_children():
    if isinstance(module, nn.Sequential):
        print(f"{name}: Sequential ({len(module)} blokk)")
    else:
        print(f"{name}: {module.__class__.__name__}")

## 2. Hook-ok használata

A **forward hook** segítségével bármely köztes réteg kimenetét "elkaphatjuk".

```python
def hook_fn(module, input, output):
    saved_features = output  # Elmentjük

handle = layer.register_forward_hook(hook_fn)
```

In [ ]:
class FeatureExtractor(nn.Module):
    """
    Jellemzők kinyerése előtanított modellből.

    Használat:
        extractor = FeatureExtractor('resnet18', 'avgpool')
        features = extractor(image_tensor)  # [batch, 512]
    """
    def __init__(self, model_name='resnet18', layer_name='avgpool'):
        super().__init__()

        # Modell betöltése
        if model_name == 'resnet18':
            self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        elif model_name == 'resnet50':
            self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.layer_name = layer_name
        self.features = None

        # Hook regisztrálása
        for name, module in self.model.named_modules():
            if name == layer_name:
                module.register_forward_hook(self._hook_fn)

        # Fagyasztás
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def _hook_fn(self, module, input, output):
        self.features = output

    def forward(self, x):
        _ = self.model(x)
        return self.features.flatten(start_dim=1)

# Teszt
extractor = FeatureExtractor('resnet18', 'avgpool')
dummy = torch.randn(1, 3, 224, 224)
features = extractor(dummy)
print(f"Bemenet: {dummy.shape} → Feature vektor: {features.shape}")

In [ ]:
# Teszt képek létrehozása
def create_test_images():
    images, labels = [], []

    shapes = [
        ('Piros kör', lambda d: d.ellipse([50,50,174,174], fill=(220,60,60))),
        ('Zöld négyzet', lambda d: d.rectangle([50,50,174,174], fill=(60,180,60))),
        ('Kék háromszög', lambda d: d.polygon([(112,40),(40,184),(184,184)], fill=(60,60,200))),
        ('Sárga csíkok', lambda d: [d.rectangle([0,y,224,y+15], fill=(230,200,50)) for y in range(0,224,30)]),
        ('Lila csíkok', lambda d: [d.rectangle([x,0,x+15,224], fill=(180,80,180)) for x in range(0,224,30)]),
        ('Sakktábla', lambda d: [d.rectangle([i*28,j*28,(i+1)*28,(j+1)*28], fill=(50,50,50))
                                 for i in range(8) for j in range(8) if (i+j)%2==0]),
    ]

    for name, draw_fn in shapes:
        img = Image.new('RGB', (224, 224), (245, 245, 250))
        draw_fn(ImageDraw.Draw(img))
        images.append(img)
        labels.append(name)

    return images, labels

images, labels = create_test_images()

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, label in zip(axes, images, labels):
    ax.imshow(img)
    ax.set_title(label, fontsize=10)
    ax.axis('off')
plt.suptitle('Teszt képek', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Jellemző-vektorok összehasonlítása

### Cosine hasonlóság

$$\text{cos}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$

- **1**: Azonos irányú vektorok (hasonló képek)
- **0**: Merőleges (nem kapcsolódó)
- **-1**: Ellentétes (ritka képeknél)

In [ ]:
# Jellemzők kinyerése
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

extractor = FeatureExtractor('resnet18', 'avgpool')

all_features = []
for img in images:
    x = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        feat = extractor(x).numpy().flatten()
    all_features.append(feat)
all_features = np.array(all_features)

print(f"Feature mátrix: {all_features.shape}")
print(f"  → {len(images)} kép, {all_features.shape[1]} dimenziós vektorok")

In [ ]:
# Hasonlóság mátrix
def cosine_sim_matrix(features):
    norm = np.linalg.norm(features, axis=1, keepdims=True)
    normalized = features / norm
    return normalized @ normalized.T

sim_matrix = cosine_sim_matrix(all_features)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(sim_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[0].set_xticks(range(len(labels)))
axes[0].set_yticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(labels, fontsize=9)

for i in range(len(labels)):
    for j in range(len(labels)):
        color = 'white' if sim_matrix[i,j] > 0.5 else 'black'
        axes[0].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                    color=color, fontsize=9)

plt.colorbar(im, ax=axes[0], label='Cosine hasonlóság')
axes[0].set_title('Feature vektorok hasonlósága', fontsize=11)

# PCA vizualizáció
pca = PCA(n_components=2)
features_2d = pca.fit_transform(all_features)

colors = plt.cm.Set1(np.linspace(0, 1, len(labels)))
for i, (x, y) in enumerate(features_2d):
    axes[1].scatter(x, y, c=[colors[i]], s=200, edgecolors='black', linewidth=2)
    axes[1].annotate(labels[i], (x, y), textcoords="offset points",
                    xytext=(0, 10), ha='center', fontsize=9)

axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('Feature vektorok 2D-ben (PCA)', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Különböző rétegek vizsgálata

| Réteg | Mit lát? | Feature dim | Használat |
|-------|----------|-------------|----------|
| `conv1` | Élek, színek | 64×112×112 | Textúra |
| `layer1` | Egyszerű minták | 64×56×56 | Stílus |
| `layer2-3` | Összetett struktúrák | 128-256 | Objektum részek |
| `layer4` | Szemantikus | 512×7×7 | Felismerés |
| `avgpool` | Globális | 512 | Hasonlóság, klaszter |

In [ ]:
# Aktivációk különböző rétegekből
class ActivationExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.eval()

        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4

    def forward(self, x):
        acts = {}
        x = self.relu(self.bn1(self.conv1(x)))
        acts['conv1'] = x.clone()
        x = self.maxpool(x)
        x = self.layer1(x); acts['layer1'] = x.clone()
        x = self.layer2(x); acts['layer2'] = x.clone()
        x = self.layer3(x); acts['layer3'] = x.clone()
        x = self.layer4(x); acts['layer4'] = x.clone()
        return acts

act_extractor = ActivationExtractor()

# Vizualizáció két képre
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for row, img_idx in enumerate([0, 3]):  # Kör és csíkok
    x = preprocess(images[img_idx]).unsqueeze(0)
    with torch.no_grad():
        activations = act_extractor(x)

    for col, (name, act) in enumerate(activations.items()):
        # Csatornák átlaga
        act_mean = act[0].mean(dim=0).numpy()
        axes[row, col].imshow(act_mean, cmap='viridis')
        if row == 0:
            axes[row, col].set_title(f'{name}\n{act.shape[2]}×{act.shape[3]}', fontsize=10)
        axes[row, col].axis('off')

    # Kép címke
    axes[row, 0].set_ylabel(labels[img_idx], fontsize=11)

plt.suptitle('Aktivációk különböző rétegekben', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Gyakorlati alkalmazások

### Kép keresés (Image Retrieval)

1. Minden képből feature vektor
2. Lekérdezés képből is feature vektor
3. Legközelebbi vektorok keresése (cosine hasonlóság)

In [ ]:
def find_similar(query_idx, features, images, labels, top_k=3):
    """Hasonló képek keresése."""
    query = features[query_idx]

    # Hasonlóságok
    sims = []
    for i, feat in enumerate(features):
        if i != query_idx:
            sim = np.dot(query, feat) / (np.linalg.norm(query) * np.linalg.norm(feat))
            sims.append((i, sim))

    sims.sort(key=lambda x: x[1], reverse=True)

    # Vizualizáció
    fig, axes = plt.subplots(1, top_k + 1, figsize=(3.5*(top_k+1), 3.5))

    axes[0].imshow(images[query_idx])
    axes[0].set_title(f'Keresés:\n{labels[query_idx]}', fontsize=10)
    axes[0].add_patch(plt.Rectangle((0,0), 223, 223, fill=False,
                                     edgecolor='blue', linewidth=4))
    axes[0].axis('off')

    for j, (idx, sim) in enumerate(sims[:top_k]):
        axes[j+1].imshow(images[idx])
        axes[j+1].set_title(f'#{j+1}: {labels[idx]}\nSim: {sim:.3f}', fontsize=10)
        axes[j+1].axis('off')

    plt.suptitle('Tartalmi alapú képkeresés', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Keresés: mi hasonlít a körhöz?
find_similar(0, all_features, images, labels)

# Keresés: mi hasonlít a csíkokhoz?
find_similar(3, all_features, images, labels)

## Összefoglalás

### Feature extraction lépései

1. **Előtanított modell** betöltése (ResNet, VGG, etc.)
2. **Hook** regisztrálása a kívánt rétegre
3. Forward pass → **feature vektor** kinyerése
4. Vektorok **összehasonlítása** (cosine hasonlóság)

### Melyik réteget használjam?

| Feladat | Réteg | Miért? |
|---------|-------|--------|
| Osztályozás | `avgpool` | Szemantikus, kompakt |
| Stílus transzfer | `layer1-3` | Textúra információ |
| Kép keresés | `avgpool` | Globális leírás |
| Objektum detekció | `layer3-4` | Lokalizáció + szemantika |

### Kód minta

```python
# Feature extraction használata
extractor = FeatureExtractor('resnet18', 'avgpool')
features = extractor(preprocessed_image)
similarity = cosine_similarity(features_a, features_b)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from typing import Dict, List, Tuple, Optional
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Feature Extraction alapjai

### A CNN mint jellemző-hierarchia

Egy tipikus CNN (pl. ResNet, VGG) az alábbi módon dolgozza fel a képet:

```
Kép (224×224×3)
    ↓ [Conv1: 7×7, stride 2]
Feature Map (112×112×64)    ← Élek, egyszerű minták
    ↓ [MaxPool + Conv blokkok]
Feature Map (56×56×64)      ← Textúrák, lokális minták
    ↓ [Conv blokkok]
Feature Map (28×28×128)     ← Összetettebb struktúrák
    ↓ [Conv blokkok]
Feature Map (14×14×256)     ← Objektum-részek
    ↓ [Conv blokkok]
Feature Map (7×7×512)       ← Szemantikus jellemzők
    ↓ [Global Average Pool]
Feature Vector (512)        ← "A kép sűrített reprezentációja"
    ↓ [Fully Connected]
Output (1000 osztály)       ← ImageNet predikció
```

### Rétegek és jellemzők összefüggése

| Réteg pozíció | Feature Map méret | Mit tanul? | Tipikus felhasználás |
|---------------|-------------------|------------|---------------------|
| Nagyon korai (conv1) | Nagy (112×112) | Élek, színek | Textúra elemzés, stílus |
| Korai (layer1-2) | Közepes (56-28) | Textúrák, minták | Lokális jellemzők |
| Középső (layer3) | Kisebb (14) | Objektum részek | Objektum detekció |
| Késői (layer4) | Kis (7) | Szemantikus | Osztályozás, hasonlóság |
| Pooling után | Vektor (512-2048) | Globális | Kép keresés, klaszterezés |

In [ ]:
# Előtanított ResNet18 betöltése és struktúrájának vizsgálata
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet18.eval()

print("ResNet18 struktúra (főbb komponensek):")
print("=" * 50)
for name, module in resnet18.named_children():
    if isinstance(module, nn.Sequential):
        print(f"{name}: Sequential ({len(module)} blokk)")
    else:
        print(f"{name}: {module.__class__.__name__}")

In [ ]:
# Részletesebb struktúra - layer1 belseje
print("\nLayer1 belső struktúra:")
print("=" * 50)
for name, module in resnet18.layer1.named_modules():
    if name:  # üres string kihagyása
        print(f"  {name}: {module.__class__.__name__}")

## 2. Hook-ok használata PyTorch-ban

A **forward hook** egy callback függvény, ami egy adott réteg forward pass-ja után fut le. Ezzel "elkaphatjuk" bármely köztes réteg kimenetét.

### Hook típusok:
- **Forward hook**: A réteg kimenetét kapja el
- **Backward hook**: A gradienst kapja el (hasznos vizualizációhoz)

### Hogyan működik?

```python
def hook_fn(module, input, output):
    # module: az aktuális réteg
    # input: a réteg bemenete (tuple)
    # output: a réteg kimenete
    saved_output = output  # elmentjük

# Hook regisztrálása
handle = layer.register_forward_hook(hook_fn)

# Forward pass - a hook automatikusan lefut
model(input)

# Hook eltávolítása (opcionális)
handle.remove()
```

In [ ]:
class FeatureExtractor(nn.Module):
    """
    Általános célú jellemző-kinyerő osztály.

    Használat:
        extractor = FeatureExtractor('resnet18', 'layer4')
        features = extractor(image_tensor)

    Paraméterek:
        model_name: 'resnet18', 'resnet50', 'vgg16', 'efficientnet_b0'
        layer_name: A kinyerni kívánt réteg neve (pl. 'layer4', 'avgpool')
        flatten: Ha True, a kimenetet 1D vektorrá lapítja
    """

    SUPPORTED_MODELS = {
        'resnet18': (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
        'resnet50': (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1),
        'vgg16': (models.vgg16, models.VGG16_Weights.IMAGENET1K_V1),
    }

    def __init__(self, model_name: str = 'resnet18',
                 layer_name: str = 'avgpool',
                 flatten: bool = True):
        super().__init__()

        if model_name not in self.SUPPORTED_MODELS:
            raise ValueError(f"Támogatott modellek: {list(self.SUPPORTED_MODELS.keys())}")

        model_fn, weights = self.SUPPORTED_MODELS[model_name]
        self.model = model_fn(weights=weights)
        self.layer_name = layer_name
        self.flatten = flatten
        self.features = None
        self._hook_handle = None

        # Hook regisztrálása a megadott rétegre
        self._register_hook()

        # Modell fagyasztása (nem tanítunk)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def _register_hook(self):
        """Hook regisztrálása a megadott rétegre."""
        found = False
        for name, module in self.model.named_modules():
            if name == self.layer_name:
                self._hook_handle = module.register_forward_hook(self._hook_fn)
                found = True
                break

        if not found:
            available = [n for n, _ in self.model.named_modules() if n]
            raise ValueError(f"Réteg '{self.layer_name}' nem található. "
                           f"Elérhető rétegek: {available[:20]}...")

    def _hook_fn(self, module, input, output):
        """Hook callback - elmenti a réteg kimenetét."""
        self.features = output

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass - visszaadja a kinyert jellemzőket."""
        # Teljes forward pass (a hook elkapja a köztes kimenetet)
        _ = self.model(x)

        if self.flatten:
            return self.features.flatten(start_dim=1)
        return self.features

    def get_feature_dim(self) -> int:
        """Visszaadja a jellemző-vektor dimenzióját."""
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            features = self.forward(dummy)
        return features.shape[1]

    def remove_hook(self):
        """Hook eltávolítása (memória felszabadítás)."""
        if self._hook_handle:
            self._hook_handle.remove()

In [ ]:
# Teszteljük a FeatureExtractor-t különböző rétegekkel
print("Feature vektorok mérete különböző rétegekből:")
print("=" * 50)

layers_to_test = ['conv1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']

for layer in layers_to_test:
    try:
        extractor = FeatureExtractor('resnet18', layer, flatten=True)
        dim = extractor.get_feature_dim()
        print(f"{layer:12} → {dim:>8,} dimenziós vektor")
        extractor.remove_hook()
    except Exception as e:
        print(f"{layer:12} → Hiba: {e}")

## 3. Különböző rétegek jellemzőinek vizsgálata

Most vizualizáljuk, mit "lát" a hálózat különböző rétegein. Ehhez létrehozunk néhány egyszerű teszt képet.

In [ ]:
def create_test_images() -> Tuple[List[Image.Image], List[str]]:
    """
    Különböző geometriai alakzatokat és textúrákat tartalmazó teszt képek.
    """
    images = []
    labels = []

    # 1. Piros kör
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.ellipse([50, 50, 174, 174], fill=(220, 60, 60), outline=(180, 40, 40), width=3)
    images.append(img)
    labels.append('Piros kör')

    # 2. Zöld négyzet
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.rectangle([50, 50, 174, 174], fill=(60, 180, 60), outline=(40, 140, 40), width=3)
    images.append(img)
    labels.append('Zöld négyzet')

    # 3. Kék háromszög
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.polygon([(112, 40), (40, 184), (184, 184)], fill=(60, 60, 200), outline=(40, 40, 160))
    images.append(img)
    labels.append('Kék háromszög')

    # 4. Sárga csíkok (vízszintes)
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    for y in range(0, 224, 30):
        draw.rectangle([0, y, 224, y+15], fill=(230, 200, 50))
    images.append(img)
    labels.append('Vízszintes csíkok')

    # 5. Függőleges csíkok
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    for x in range(0, 224, 30):
        draw.rectangle([x, 0, x+15, 224], fill=(200, 100, 200))
    images.append(img)
    labels.append('Függőleges csíkok')

    # 6. Sakktábla minta
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    cell_size = 28
    for i in range(8):
        for j in range(8):
            if (i + j) % 2 == 0:
                draw.rectangle([i*cell_size, j*cell_size,
                              (i+1)*cell_size, (j+1)*cell_size],
                             fill=(50, 50, 50))
    images.append(img)
    labels.append('Sakktábla')

    # 7. Koncentrikus körök
    img = Image.new('RGB', (224, 224), (245, 245, 250))
    draw = ImageDraw.Draw(img)
    center = 112
    for r in range(100, 10, -20):
        color = (int(255 * r/100), int(100 * (1-r/100)), int(200 * r/100))
        draw.ellipse([center-r, center-r, center+r, center+r], outline=color, width=8)
    images.append(img)
    labels.append('Koncentrikus körök')

    # 8. Gradiens
    img_array = np.zeros((224, 224, 3), dtype=np.uint8)
    for x in range(224):
        img_array[:, x, 0] = int(255 * x / 224)
        img_array[:, x, 2] = int(255 * (1 - x / 224))
    images.append(Image.fromarray(img_array))
    labels.append('Gradiens')

    return images, labels

# Képek létrehozása és megjelenítése
test_images, test_labels = create_test_images()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, label in zip(axes.flat, test_images, test_labels):
    ax.imshow(img)
    ax.set_title(label, fontsize=11)
    ax.axis('off')
plt.suptitle('Teszt képek a feature extraction-höz', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Előfeldolgozás ImageNet normalizálással
preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

def extract_features_from_images(images: List[Image.Image],
                                  layer_name: str = 'avgpool') -> np.ndarray:
    """
    Jellemzők kinyerése képek listájából.
    """
    extractor = FeatureExtractor('resnet18', layer_name, flatten=True)

    features = []
    for img in images:
        x = preprocess(img).unsqueeze(0)
        with torch.no_grad():
            feat = extractor(x)
        features.append(feat.numpy().flatten())

    extractor.remove_hook()
    return np.array(features)

# Jellemzők kinyerése az avgpool rétegből
features = extract_features_from_images(test_images, 'avgpool')
print(f"Kinyert jellemzők alakja: {features.shape}")
print(f"  - {len(test_images)} kép")
print(f"  - {features.shape[1]} dimenziós feature vektor képenként")

## 4. Jellemző-vektorok összehasonlítása

### Cosine hasonlóság

A jellemző-vektorok összehasonlítására leggyakrabban a **cosine hasonlóságot** használjuk:

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

Ez a mérték:
- **1**: Azonos irányú vektorok (nagyon hasonló képek)
- **0**: Merőleges vektorok (nem kapcsolódó képek)
- **-1**: Ellentétes irányú vektorok (ritkán fordul elő képeknél)

In [ ]:
def cosine_similarity_matrix(features: np.ndarray) -> np.ndarray:
    """
    Páronkénti cosine hasonlóság mátrix számítása.
    """
    # L2 normalizálás
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    normalized = features / norms

    # Cosine hasonlóság = skaláris szorzat normalizált vektorok között
    similarity = normalized @ normalized.T
    return similarity

# Hasonlóság mátrix
sim_matrix = cosine_similarity_matrix(features)

# Vizualizáció
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
im = axes[0].imshow(sim_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[0].set_xticks(range(len(test_labels)))
axes[0].set_yticks(range(len(test_labels)))
axes[0].set_xticklabels(test_labels, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(test_labels, fontsize=9)

# Értékek beírása
for i in range(len(test_labels)):
    for j in range(len(test_labels)):
        color = 'white' if sim_matrix[i, j] > 0.5 else 'black'
        axes[0].text(j, i, f'{sim_matrix[i,j]:.2f}',
                    ha='center', va='center', color=color, fontsize=8)

plt.colorbar(im, ax=axes[0], label='Cosine hasonlóság')
axes[0].set_title('Jellemző-vektorok hasonlósága (ResNet18 avgpool)', fontsize=12)

# Dendrogram-szerű rendezés - leginkább hasonló párok
pairs = []
for i in range(len(test_labels)):
    for j in range(i+1, len(test_labels)):
        pairs.append((sim_matrix[i,j], test_labels[i], test_labels[j]))

pairs.sort(reverse=True)

y_pos = np.arange(len(pairs))
colors = plt.cm.RdYlBu_r([p[0] for p in pairs])
axes[1].barh(y_pos, [p[0] for p in pairs], color=colors)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([f"{p[1]} ↔ {p[2]}" for p in pairs], fontsize=8)
axes[1].set_xlabel('Cosine hasonlóság')
axes[1].set_title('Képpárok hasonlóság szerinti sorrendben', fontsize=12)
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Jellemzők vizualizálása 2D-ben (t-SNE és PCA)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features)

scatter = axes[0].scatter(features_pca[:, 0], features_pca[:, 1],
                         c=range(len(test_labels)), cmap='tab10', s=200)
for i, label in enumerate(test_labels):
    axes[0].annotate(label, (features_pca[i, 0], features_pca[i, 1]),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=9)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variancia)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variancia)')
axes[0].set_title('PCA vetítés', fontsize=12)

# t-SNE (több mintával lenne jobb, de működik)
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
features_tsne = tsne.fit_transform(features)

axes[1].scatter(features_tsne[:, 0], features_tsne[:, 1],
               c=range(len(test_labels)), cmap='tab10', s=200)
for i, label in enumerate(test_labels):
    axes[1].annotate(label, (features_tsne[i, 0], features_tsne[i, 1]),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=9)
axes[1].set_xlabel('t-SNE dimenzió 1')
axes[1].set_ylabel('t-SNE dimenzió 2')
axes[1].set_title('t-SNE vetítés', fontsize=12)

plt.suptitle('Feature vektorok 2D reprezentációja', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Gyakorlati alkalmazások

### 5.1 Kép keresés (Image Retrieval)

A feature extraction egyik leggyakoribb alkalmazása a **tartalmi alapú képkeresés**. A lépések:

1. Az adatbázis minden képéből feature vektort számolunk
2. A keresési lekérdezés képéből is feature vektort készítünk
3. Megkeressük a legközelebbi vektorokat (cosine hasonlóság vagy L2 távolság)

In [ ]:
def find_similar_images(query_idx: int,
                        features: np.ndarray,
                        images: List[Image.Image],
                        labels: List[str],
                        top_k: int = 3):
    """
    Hasonló képek keresése egy lekérdezés alapján.
    """
    query_feature = features[query_idx]

    # Cosine hasonlóságok
    similarities = []
    for i, feat in enumerate(features):
        if i != query_idx:
            sim = np.dot(query_feature, feat) / (
                np.linalg.norm(query_feature) * np.linalg.norm(feat)
            )
            similarities.append((i, sim))

    # Rendezés csökkenő sorrendben
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Vizualizáció
    fig, axes = plt.subplots(1, top_k + 1, figsize=(4*(top_k+1), 4))

    # Lekérdezés
    axes[0].imshow(images[query_idx])
    axes[0].set_title(f'Lekérdezés:\n{labels[query_idx]}', fontsize=11)
    axes[0].axis('off')
    axes[0].add_patch(plt.Rectangle((0, 0), 223, 223,
                                    fill=False, edgecolor='blue', linewidth=4))

    # Top-K találatok
    for j, (idx, sim) in enumerate(similarities[:top_k]):
        axes[j+1].imshow(images[idx])
        axes[j+1].set_title(f'#{j+1}: {labels[idx]}\nHasonlóság: {sim:.3f}', fontsize=10)
        axes[j+1].axis('off')

    plt.suptitle('Tartalmi alapú képkeresés eredménye', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Keresés: mi hasonlít a piros körhöz?
find_similar_images(0, features, test_images, test_labels, top_k=4)

In [ ]:
# Keresés a csíkos mintákra
find_similar_images(3, features, test_images, test_labels, top_k=4)

### 5.2 Kép klaszterezés

A feature vektorok alapján automatikusan csoportosíthatjuk a képeket.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Hierarchikus klaszterezés
linkage_matrix = linkage(features, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dendrogram
dendrogram(linkage_matrix, labels=test_labels, ax=axes[0],
          leaf_rotation=45, leaf_font_size=10)
axes[0].set_title('Hierarchikus klaszterezés (dendrogram)', fontsize=12)
axes[0].set_ylabel('Távolság')

# Klaszterek (3 csoportba)
clustering = AgglomerativeClustering(n_clusters=3)
clusters = clustering.fit_predict(features)

colors = plt.cm.Set1(clusters)
features_pca = PCA(n_components=2).fit_transform(features)

for i, (label, color) in enumerate(zip(test_labels, colors)):
    axes[1].scatter(features_pca[i, 0], features_pca[i, 1],
                   c=[color], s=300, edgecolors='black', linewidth=2)
    axes[1].annotate(label, (features_pca[i, 0], features_pca[i, 1]),
                    textcoords="offset points", xytext=(0, 12),
                    ha='center', fontsize=9)

axes[1].set_title('Klaszterek (3 csoport) PCA térben', fontsize=12)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print("\nKlaszter hozzárendelések:")
for cluster_id in range(3):
    members = [test_labels[i] for i in range(len(test_labels)) if clusters[i] == cluster_id]
    print(f"  Klaszter {cluster_id}: {', '.join(members)}")

## 6. Többrétegű jellemző-kinyerés

Néha hasznos **több rétegből** is kinyerni jellemzőket. A korai rétegek lokális, textúra-jellegű információt adnak, míg a késői rétegek szemantikusabbak.

In [ ]:
class MultiLayerFeatureExtractor(nn.Module):
    """
    Több rétegből is kinyeri a jellemzőket.
    """

    def __init__(self, layer_names: List[str] = ['layer2', 'layer3', 'layer4']):
        super().__init__()

        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.layer_names = layer_names
        self.features = {}

        # Hook regisztrálása minden megadott rétegre
        for name, module in self.model.named_modules():
            if name in layer_names:
                module.register_forward_hook(self._make_hook(name))

        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def _make_hook(self, name):
        """Hook factory - minden réteghez saját hook."""
        def hook(module, input, output):
            self.features[name] = output
        return hook

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        self.features = {}  # Törlés
        _ = self.model(x)

        # Global average pooling minden feature map-re
        pooled_features = {}
        for name, feat in self.features.items():
            pooled = F.adaptive_avg_pool2d(feat, 1).flatten(start_dim=1)
            pooled_features[name] = pooled

        return pooled_features

    def get_concatenated_features(self, x: torch.Tensor) -> torch.Tensor:
        """Az összes réteg jellemzőinek konkatenálása."""
        features = self.forward(x)
        return torch.cat(list(features.values()), dim=1)

In [ ]:
# Többrétegű jellemzők vizsgálata
multi_extractor = MultiLayerFeatureExtractor(['layer1', 'layer2', 'layer3', 'layer4'])

# Egy kép jellemzői minden rétegből
x = preprocess(test_images[0]).unsqueeze(0)
with torch.no_grad():
    multi_features = multi_extractor(x)

print("Többrétegű jellemzők méretei:")
print("=" * 40)
total_dim = 0
for name, feat in multi_features.items():
    print(f"  {name}: {feat.shape[1]} dimenzió")
    total_dim += feat.shape[1]

print(f"\nKonkatenált vektor: {total_dim} dimenzió")

In [ ]:
# Aktivációk vizualizálása különböző rétegekből

# Hook nélküli direkt megközelítés a feature map-ekhez
class ActivationVisualizer(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.eval()

        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4

    def forward(self, x):
        activations = {}

        x = self.conv1(x)
        activations['conv1'] = x.clone()

        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        activations['layer1'] = x.clone()

        x = self.layer2(x)
        activations['layer2'] = x.clone()

        x = self.layer3(x)
        activations['layer3'] = x.clone()

        x = self.layer4(x)
        activations['layer4'] = x.clone()

        return activations

vis = ActivationVisualizer()

# Aktivációk két képre
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for row, img_idx in enumerate([0, 3]):  # Kör és csíkok
    x = preprocess(test_images[img_idx]).unsqueeze(0)
    with torch.no_grad():
        activations = vis(x)

    # Eredeti kép
    axes[row, 0].imshow(test_images[img_idx])
    axes[row, 0].set_title(f'Eredeti:\n{test_labels[img_idx]}', fontsize=10)
    axes[row, 0].axis('off')

    # Aktivációk
    for col, (name, act) in enumerate(activations.items()):
        # Átlagoljuk a csatornákat
        act_mean = act[0].mean(dim=0).numpy()
        axes[row, col+1].imshow(act_mean, cmap='viridis')
        axes[row, col+1].set_title(f'{name}\n{act.shape[2]}×{act.shape[3]}', fontsize=10)
        axes[row, col+1].axis('off')

plt.suptitle('Aktivációk különböző rétegekben (csatornák átlaga)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Összefoglalás

### Mit tanultunk?

1. **Feature extraction** = Előtanított hálózat köztes rétegeinek felhasználása
2. **Hook-ok** segítségével bármely réteg kimenetét elkaphatjuk
3. **Korai rétegek** = lokális, textúra jellemzők
4. **Késői rétegek** = szemantikus, magas szintű jellemzők
5. **Cosine hasonlóság** = jellemző-vektorok összehasonlítása

### Gyakorlati tippek

| Feladat | Ajánlott réteg | Miért? |
|---------|----------------|--------|
| Képosztályozás | `avgpool` / `layer4` | Szemantikus információ |
| Stílus transzfer | `layer1-3` | Textúra jellemzők |
| Objektum detekció | `layer3-4` | Lokalizációs információ |
| Kép keresés | `avgpool` | Kompakt, globális leírás |

### További olvasmányok

- [Feature extraction torchvision dokumentáció](https://pytorch.org/vision/stable/feature_extraction.html)
- [Visualizing CNN features](https://distill.pub/2017/feature-visualization/)